# Notebook 12 — Bring Your Own PDE

**What you'll learn:**
- How to solve your own equation end-to-end with Lang-PINN
- Customizing the generated code
- Comparing agent recommendations with your domain expertise
- The full workflow: describe → verify → generate → customize → train

**Prerequisites:** Notebooks 10-11

**Time:** ~40 minutes

## The Workflow

```
1. Describe your PDE (build a PDESpec)
2. Verify with SymPy
3. Get architecture recommendation
4. Generate code
5. Review and customize
6. Run it
```

We'll walk through this with a real example: **the advection equation**.

$$u_t + c \cdot u_x = 0, \quad x \in [0, 2\pi], \quad t \in [0, 2]$$

with initial condition $u(x, 0) = \sin(x)$ and periodic boundaries.

The exact solution is $u(x, t) = \sin(x - ct)$ — a traveling wave.

## Step 1: Define Your PDE

In [ ]:
import numpy as np
from lang_pinn import PDESpec

advection = PDESpec(
    name="Advection Equation",
    equation="u_t + c*u_x = 0",
    independent_vars=["x", "t"],
    dependent_var="u",
    order=1,
    spatial_dim=1,
    domain={"x": (0.0, 2 * np.pi), "t": (0.0, 2.0)},
    initial_conditions=["u(x, 0) = sin(x)"],
    boundary_conditions=["periodic in x"],
    parameters={"c": 1.0},
    is_linear=True,
    has_periodic_bc=True,
)

print(f"Defined: {advection.name}")
print(f"Equation: {advection.equation}")
print(f"Domain: {advection.domain}")

## Step 2: Verify

In [ ]:
from lang_pinn import verify_spec

issues = verify_spec(advection)
if issues:
    print("Issues found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All checks passed!")

## Step 3: Get Architecture Recommendation

In [ ]:
from lang_pinn import PINNAgent

pinn_agent = PINNAgent()
arch = pinn_agent.recommend(advection)

print("Recommended architecture:")
print(f"  Network: {arch.hidden_layers}x{arch.hidden_neurons} {arch.activation}")
print(f"  Epochs: {arch.epochs}")
print(f"  LR: {arch.learning_rate}")
print(f"  Collocation: {arch.n_collocation}")
print(f"  Loss weights: {arch.loss_weights}")
print(f"  Ansatz: {arch.use_ansatz}")
print(f"\nReasoning: {arch.reasoning}")

### Domain Expertise Check

This is where your knowledge matters. The agent gives a good starting point, but you know your problem:

- The advection equation is linear and smooth → `tanh` activation is fine
- With periodic BCs, we might want to increase collocation near boundaries
- The wave speed `c=1` means the solution translates — nothing too stiff

The recommendation looks reasonable. Let's customize if needed:

In [ ]:
from lang_pinn import ArchitectureRec

# You can override any recommendation
custom_arch = ArchitectureRec(
    input_dim=arch.input_dim,
    output_dim=arch.output_dim,
    hidden_layers=4,         # keep the recommendation
    hidden_neurons=64,       # keep the recommendation
    activation="tanh",
    learning_rate=1e-3,
    epochs=5000,             # reduce for demo
    loss_weights={"ic": 10.0, "bc": 5.0, "physics": 1.0},  # IC matters more
    n_collocation=3000,      # increase slightly for periodic domain
    reasoning="Custom: emphasize IC/BC for advection with periodic boundaries",
)

print(f"Custom architecture: {custom_arch.hidden_layers}x{custom_arch.hidden_neurons}")
print(f"Custom weights: {custom_arch.loss_weights}")

## Step 4: Generate Code

In [ ]:
from lang_pinn import CodeAgent

code_agent = CodeAgent()
code = code_agent.generate(advection, custom_arch, use_llm=False)

print(f"Generated {code.count(chr(10)) + 1} lines of code")
print("\nKey sections:")
for line in code.split("\n"):
    if line.startswith("def ") or line.startswith("# ----"):
        print(f"  {line}")

## Step 5: Review the Generated Code

Let's look at what was generated:

In [ ]:
print(code)

## Step 5b: Customize the Residual

The template generates a **placeholder residual** (`# TODO: complete residual`). For production use, you'd fill this in. But even the template gives you the right derivative computation — you just need to assemble the equation.

For the advection equation $u_t + c \cdot u_x = 0$, the residual is:

```python
residual = u_t + c * u_x
```

In template mode, the Code Agent can't know the exact equation structure (that requires the LLM). But it gives you the scaffolding — model, collocation points, derivatives, training loop — and you fill in one line.

## The Big Picture: What Lang-PINN Saves You

Without Lang-PINN, solving a new PDE requires:
1. Deciding on network architecture (layers, neurons, activation) — **PINN Agent does this**
2. Writing collocation point setup — **Code Agent does this**
3. Writing derivative computation code — **Code Agent does this**
4. Setting up the training loop — **Code Agent does this**
5. Adding monitoring and quality evaluation — **Code Agent does this**

You still need to:
1. Know what equation you're solving (obviously)
2. Fill in the residual expression (template mode) or verify LLM output (code-agent/hybrid mode)
3. Validate against known solutions when available

**Lang-PINN doesn't replace your physics knowledge — it handles the boilerplate so you can focus on the physics.**

## Exercise: Try Your Own PDE

Build a `PDESpec` for one of these (or your own):

1. **Heat equation**: $u_t = \alpha \cdot u_{xx}$ on $[0, 1] \times [0, 0.5]$
2. **Wave equation**: $u_{tt} = c^2 \cdot u_{xx}$ on $[0, \pi] \times [0, 2]$
3. **Poisson equation**: $u_{xx} + u_{yy} = f(x,y)$ on $[0, 1]^2$ (steady state)

Then:
- Verify with SymPy
- Get architecture recommendation
- Generate code
- Compare the recommendation with what you'd choose

In [ ]:
# Your turn!
# my_spec = PDESpec(
#     name="...",
#     equation="...",
#     ...
# )
# issues = verify_spec(my_spec)
# arch = PINNAgent().recommend(my_spec)
# code = CodeAgent().generate(my_spec, arch)

## Summary: The Complete PINN Curriculum

You've now completed the full learning path:

| Notebook | Topic | Key Skill |
|----------|-------|----------|
| 01 | What are PINNs? | Motivation, landscape |
| 02 | Automatic differentiation | `torch.autograd.grad` |
| 03 | First PINN from scratch | Raw PyTorch PINN |
| 04 | Data vs Physics vs Hybrid | Loss function design |
| 05 | PDEs and boundary conditions | Multi-term losses |
| 06 | Training tricks | Ansatz, weighting, scheduling |
| 07 | Parametric and inverse | Parameters as inputs |
| 08 | Honest assessment | When (not) to use PINNs |
| 09 | Inverse Navier-Stokes | Advanced: Re inference |
| **10** | **Lang-PINN intro** | **3 agents, 3 modes** |
| **11** | **Hybrid mode** | **Feedback loop, quality scoring** |
| **12** | **Bring your own PDE** | **End-to-end workflow** |

From first principles to LLM-guided automation in 12 notebooks. You now have the tools to solve differential equations with neural networks — and the judgment to know when to use them.

**Happy solving!**